In [17]:
#1
#구글 드라이브 연결
#kagggle에서 가져온 2개의 CSV 파일 "tmdb_5000_movies.csv", "tmdb_5000_credits.csv" 병합
import pandas as pd
from google.colab import drive


drive.mount('/content/drive')

movies_path = '/content/drive/MyDrive/지추텍/tmdb_5000_movies.csv'
credits_path = '/content/drive/MyDrive/지추텍/tmdb_5000_credits.csv'


movies_df = pd.read_csv(movies_path)
credits_df = pd.read_csv(credits_path)

# 'title' 컬럼을 기준으로 데이터프레임 병합 (Inner Join)
merged_df = pd.merge(movies_df, credits_df, on='title')

# credits_df의 movie_id를 대표 id 컬럼으로 지정
merged_df['id'] = merged_df['movie_id']

print("--- [STEP 1-1] 구글 드라이브 연동 및 제목(title) 기준 병합 완료 ---")
print(f"합쳐진 데이터셋의 총 영화 개수: {len(merged_df)}개")
print(f"결합된 컬럼 목록:\n{list(merged_df.columns)}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
--- [STEP 1-1] 구글 드라이브 연동 및 제목(title) 기준 병합 완료 ---
합쳐진 데이터셋의 총 영화 개수: 4809개
결합된 컬럼 목록:
['budget', 'genres', 'homepage', 'id', 'keywords', 'original_language', 'original_title', 'overview', 'popularity', 'production_companies', 'production_countries', 'release_date', 'revenue', 'runtime', 'spoken_languages', 'status', 'tagline', 'title', 'vote_average', 'vote_count', 'movie_id', 'cast', 'crew']


In [22]:
#2
#JSON 형태의 문자열에서 장르, 키워드, 배우, 감독이름을 단어 단위로 가져오는 함수 정의.
#뽑아낸 단어들을 소문자로 바꾸고 띄어쓰기를 언더바(_)로 연결하여 데이터 규격 맞춤.
#토픽 모델링 학습(LDA)을 위해 영화 줄거리(overview),
import ast
 #안전하게 문자열을 파이썬 리스트/딕셔너리로 변환하는 함수 정의
def parse_json_column(json_str, key_name='name', top_n=None):
    try:
        data_list = ast.literal_eval(json_str)
        if top_n:  # 상위 N명만 추출
            data_list = data_list[:top_n]
        # 소문자 변환 및 공백을 언더바로 처리 (Prolog Fact 규격)
        return [item[key_name].lower().replace(" ", "_") for item in data_list]
    except:
        return []

# 크루(crew) 데이터에서 감독(Director)만 추출하는 함수 정의
def get_director(json_str):
    try:
        data_list = ast.literal_eval(json_str)
        for item in data_list:
            if item.get('job') == 'Director':
                return [item['name'].lower().replace(" ", "_")]
        return []
    except:
        return []

# 각 컬럼에 파싱 함수 적용
merged_df['genres_clean'] = merged_df['genres'].apply(lambda x: parse_json_column(x))
merged_df['keywords_clean'] = merged_df['keywords'].apply(lambda x: parse_json_column(x))
merged_df['cast_clean'] = merged_df['cast'].apply(lambda x: parse_json_column(x, top_n=3)) # 주연 배우 상위 3명 > 조절 가능
merged_df['director_clean'] = merged_df['crew'].apply(get_director) # 감독

# 결측치 처리 및 LDA 토픽 추출용 통합 텍스트 컬럼 생성 (줄거리 + 키워드)
merged_df['overview'] = merged_df['overview'].fillna('')
merged_df['total_features'] = merged_df['overview'] + " " + merged_df['keywords_clean'].apply(lambda x: " ".join(x))

print("--- 문자열 데이터 파싱 및 텍스트 전처리 완료 ---")
# 가공이 잘 되었는지 상위 3개 행 확인
print(merged_df[['title', 'genres_clean', 'cast_clean', 'director_clean']].head(3))

--- 문자열 데이터 파싱 및 텍스트 전처리 완료 ---
                                      title  \
0                                    Avatar   
1  Pirates of the Caribbean: At World's End   
2                                   Spectre   

                                    genres_clean  \
0  [action, adventure, fantasy, science_fiction]   
1                   [adventure, fantasy, action]   
2                     [action, adventure, crime]   

                                         cast_clean    director_clean  
0  [sam_worthington, zoe_saldana, sigourney_weaver]   [james_cameron]  
1     [johnny_depp, orlando_bloom, keira_knightley]  [gore_verbinski]  
2      [daniel_craig, christoph_waltz, léa_seydoux]      [sam_mendes]  


In [19]:
#3
#LDA 토픽 분석: 전체 영화를 10개의 주제로 자동 분류, 각 영화가 어떤 토픽에 가까운지 확률 계산
#TF-IDF 키워드 추출: 그 영화만의 개성 있는 핵심 단어 상위 5개를 추출
#테이블 분리 및 컬럼명 자동화
#데이터프레임1: "영화 기본 정보+TF-IDF 키워드 5개"
#데이터프레임2: "영화id + topic_0~topic_9의 토픽 확률"
#한글 제거 전처리
import numpy as np
import pandas as pd
import re
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation

# [LDA 분석] - 단어 빈도 기반 잠재 토픽 확률 추출
count_vectorizer = CountVectorizer(stop_words='english', max_features=2000)
count_data = count_vectorizer.fit_transform(merged_df['total_features'])

NUM_TOPICS = 10   # 토픽 개수 10개로 설정 > 조절 가능
lda_model = LatentDirichletAllocation(n_components=NUM_TOPICS, random_state=42)
lda_outputs = lda_model.fit_transform(count_data)

# [TF-IDF 분석] - 영화별 중요 고유 키워드 상위 n개 추출
tfidf_vectorizer = TfidfVectorizer(stop_words='english', max_features=2000)
tfidf_matrix = tfidf_vectorizer.fit_transform(merged_df['total_features'])
tfidf_feature_names = tfidf_vectorizer.get_feature_names_out()

top_keywords_list = []
for i in range(len(merged_df)):
    row_scores = tfidf_matrix[i].toarray()[0]
    top_indices = row_scores.argsort()[::-1][:5] # 점수 높은 순 상위 5개 > 조절 가능
    top_words = [tfidf_feature_names[idx] for idx in top_indices if row_scores[idx] > 0]
    top_keywords_list.append(top_words)

# 전처리 완료된 통합 데이터셋 (TF-IDF 결과 포함)
base_columns = [
    'id', 'title', 'release_date', 'vote_average', 'runtime', 'popularity',
    'genres_clean', 'cast_clean', 'director_clean', 'overview'
]
df_a_integrated = merged_df[base_columns].copy()
df_a_integrated['keywords_tfidf'] = top_keywords_list  # TF-IDF 단어 리스트 추가

# LDA 토픽 모델링 결과 매핑 테이블
# 10개의 토픽 이름을 topic_0 ~ topic_9로 설정.
topic_labels = [f'topic_{i}' for i in range(NUM_TOPICS)]

# 생성한 토픽 이름(topic_0 ~ topic_9)을 컬럼명으로 사용하여 데이터프레임 생성.
df_b_topics = pd.DataFrame(lda_outputs, columns=topic_labels)

# 맨 앞에 영화 고유 ID와 제목을 삽입.
df_b_topics.insert(0, 'id', merged_df['id'].values)
df_b_topics.insert(1, 'title', merged_df['title'].values)



# 한글 제거 기능
def remove_korean(text):
    if isinstance(text, str):
        return re.sub(r'[ㄱ-ㅎㅏ-ㅣ가-힣]+', '', text).strip()
    return text

# A 테이블과 B 테이블에서 한글을 싹 제거합니다.
df_a_integrated = df_a_integrated.map(remove_korean)



In [20]:
#4
#데이터프레임 1,2 구글 드라이브에 CSV 파일로 저장
# 구글 드라이브 내 '지추텍' 폴더 저장 경로 설정
path_file_a = '/content/drive/MyDrive/지추텍/tmdb_final_preprocessed.csv'
path_file_b = '/content/drive/MyDrive/지추텍/tmdb_lda_topic_mapping.csv'

# 한글 깨짐 방지(utf-8-sig)를 적용하여 각각 저장
df_a_integrated.to_csv(path_file_a, index=False, encoding='utf-8-sig')
df_b_topics.to_csv(path_file_b, index=False, encoding='utf-8-sig')

print(f"▶ A 파일 (메타데이터 및 TF-IDF 키워드): {path_file_a}")
print(f"▶ B 파일 (해석된 10개 LDA 토픽 점수): {path_file_b}")

▶ A 파일 (메타데이터 및 TF-IDF 키워드): /content/drive/MyDrive/지추텍/tmdb_final_preprocessed.csv
▶ B 파일 (해석된 10개 LDA 토픽 점수): /content/drive/MyDrive/지추텍/tmdb_lda_topic_mapping.csv


In [23]:
#5
#분류한 topic_0~topic_9이 무슨 주제인지 확인
# lda_model과 count_vectorizer가 이전 셀에서 학습된 상태여야 합니다.
import numpy as np

# 벡터라이저에 등록된 전체 단어 목록 가져오기
feature_names = count_vectorizer.get_feature_names_out()

# 10개 토픽 각각을 구성하는 상위 15개 핵심 단어 출력
print("=== LDA 토픽별 핵심 단어 ===")
for topic_idx, topic in enumerate(lda_model.components_):
    # 각 토픽 내부에서 가중치가 높은 단어의 인덱스를 역순으로 정렬
    top_features_ind = topic.argsort()[::-1][:15]  #상위 15개 < 조절 가능
    top_features = [feature_names[i] for i in top_features_ind]

    print(f"\n[topic_{topic_idx}]")
    print(" , ".join(top_features))

=== LDA 토픽별 핵심 단어 ===

[topic_0]
family , young , film , father , old , world , new , year , son , life , man , home , story , mother , parents

[topic_1]
war , world , army , battle , fight , soldier , king , american , forces , princess , queen , evil , world_war_ii , story , ii

[topic_2]
story , life , world , film , young , man , love , based , way , journey , family , children , faith , jack , set

[topic_3]
murder , family , man , young , revenge , city , killer , violence , father , town , police , sheriff , woman , gang , crime

[topic_4]
love , new , life , york , town , hotel , comedy , wife , divorce , daughter , husband , school , finds , soon , gets

[topic_5]
love , friends , life , friend , school , girl , best , family , young , man , high , woman_director , mother , teenager , independent_film

[topic_6]
year , old , home , group , friends , witch , town , evil , based_on_novel , small , help , lake , discover , new , james

[topic_7]
life , school , college , drug , 